# Logit Lens — what does each layer predict?

Peek inside the residual stream at every layer and decode it through the model's own unembedding.
Each layer gets a distribution over the vocabulary: you see the model's "best guess" evolve from
nonsense near the embedding to a sharp prediction near the top.

**Origin**: nostalgebraist 2020, *interpreting GPT: the logit lens* — [LessWrong post](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens).
Modern multi-family handling follows LogitLens4LLMs (Zhenyu 2025, [arxiv:2503.11667](https://arxiv.org/abs/2503.11667), [github](https://github.com/zhenyu-02/LogitLens4LLMs)).

**Five lines of code**:
```python
hs = model(ids, output_hidden_states=True).hidden_states   # (L+1, B, T, D)
for h in hs:
    logits = model.lm_head(model.model.norm(h))            # reuse final LN + unembed
    probs  = logits.softmax(-1)
    topk   = probs[:, -1].topk(5)                          # top-5 at last position
```

Runtime: ~5 min on a T4 with Gemma-2-2B. Pure `transformers` — no TransformerLens.

In [ ]:
!pip install -q -U transformers accelerate safetensors matplotlib tqdm

## Config

Swap `MODEL_ID` for any HF decoder-only model — the lens code below auto-detects the Llama/Gemma/Qwen
(`model.model.norm` + `model.lm_head`) vs GPT-2 (`model.transformer.ln_f` + `model.transformer.wte`) layout.

In [ ]:
MODEL_ID   = 'google/gemma-2-2b'
PROMPT     = 'The capital of France is'
MAX_TOKENS = 16       # forced-continuation length for the heatmap
TOP_K      = 5        # top predicted tokens per layer
DTYPE      = 'bfloat16'

## Load model

We ask for `output_hidden_states=True` on the forward pass so we get every layer's residual stream
as a tuple of `(n_layers + 1, batch, seq, d_model)` tensors — index 0 is the embedding, index `L`
is the final pre-unembed residual.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

dtype_map = {'bfloat16': torch.bfloat16, 'float16': torch.float16, 'float32': torch.float32}
torch_dtype = dtype_map[DTYPE]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,
    attn_implementation='sdpa',
    device_map='cuda',
)
model.eval()

print(f'Model: {MODEL_ID}')
print(f'Device: {next(model.parameters()).device}, dtype: {next(model.parameters()).dtype}')
print(f'Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B')

## Apply the lens

Fetch hidden states from every layer, then reuse the model's own final layer-norm + unembedding
to turn each residual into a next-token distribution. The same operation the final layer performs
— just applied early.

In [ ]:
def get_final_ln_and_unembed(model):
    """Return (final_ln, lm_head) for Llama/Gemma/Qwen and GPT-2 style models."""
    # Llama / Gemma / Qwen / Mistral / Phi layout
    if hasattr(model, 'model') and hasattr(model.model, 'norm'):
        final_ln = model.model.norm
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'ln_f'):
        final_ln = model.transformer.ln_f
    # Gemma-2 multimodal / language_model nesting
    elif hasattr(model, 'model') and hasattr(model.model, 'language_model') \
         and hasattr(model.model.language_model, 'norm'):
        final_ln = model.model.language_model.norm
    else:
        raise RuntimeError('Could not locate final layer norm; inspect model structure.')

    # Unembedding head
    if hasattr(model, 'lm_head') and model.lm_head is not None:
        lm_head = model.lm_head
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'wte'):
        # GPT-2 ties wte; use it as the unembed (x @ wte.weight.T)
        lm_head = lambda x: x @ model.transformer.wte.weight.T
    else:
        raise RuntimeError('Could not locate unembedding (lm_head / wte).')

    return final_ln, lm_head


def logit_lens(hidden_l, model):
    """Project a residual-stream tensor through the model's own final LN + unembed."""
    final_ln, lm_head = get_final_ln_and_unembed(model)
    h_normed = final_ln(hidden_l)
    logits = lm_head(h_normed)
    return logits.softmax(-1)


# Run a forward pass with hidden states enabled
inputs = tokenizer(PROMPT, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)

hidden_states = out.hidden_states    # tuple of length n_layers + 1
n_layers = len(hidden_states) - 1
print(f'Hidden-state tuple length: {len(hidden_states)}  (embedding + {n_layers} transformer layers)')
print(f'Per-layer shape: {tuple(hidden_states[0].shape)}   # (batch, seq, d_model)')

## Per-layer top-k at the last token

For each layer, take the residual at the **last prompt position** and print the top-K tokens the lens
would decode. Early layers tend to show surface/frequency tokens; late layers converge to the actual answer.

In [ ]:
rows = []
with torch.no_grad():
    for layer_idx, h in enumerate(hidden_states):
        probs = logit_lens(h, model)            # (B, T, V)
        last_probs = probs[0, -1]               # (V,) — distribution at last prompt position
        top = last_probs.topk(TOP_K)
        toks = [tokenizer.decode([t.item()]).replace('\n', '\\n') for t in top.indices]
        probs_list = [f'{p.item():.3f}' for p in top.values]
        rows.append((layer_idx, toks, probs_list))

# Pretty table
label = lambda i: 'emb' if i == 0 else ('final' if i == n_layers else f'L{i:02d}')
header_tok = ' | '.join(f'top{i+1:>2}'.center(14) for i in range(TOP_K))
print(f'{"layer":>6}  {header_tok}')
print('-' * (8 + 16 * TOP_K))
for layer_idx, toks, probs_list in rows:
    cells = ' | '.join(f'{repr(tk)[:10]:>8}({p})' for tk, p in zip(toks, probs_list))
    print(f'{label(layer_idx):>6}  {cells}')

## Heatmap — probability of the target token at every (layer, position)

We force a greedy continuation of the prompt for `MAX_TOKENS`, then for every position `t` in the full
sequence we track the probability the lens assigns — **at every layer** — to the token that actually
ends up at position `t + 1`. Bright horizontal bands mark the layers where the model "commits" to its
prediction. Saved as `logit_lens_heatmap.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1) Forced greedy continuation to build a ground-truth target sequence
with torch.no_grad():
    gen = model.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
full_ids = gen[0]                                    # (T_full,)
print('Continuation:', tokenizer.decode(full_ids))

# 2) Forward pass on the full sequence to get hidden states at every position
with torch.no_grad():
    out_full = model(full_ids.unsqueeze(0), output_hidden_states=True)
hs_full = out_full.hidden_states                     # tuple (L+1) of (1, T_full, D)
T_full = full_ids.shape[0]

# 3) For each layer and each position t in [0, T_full-2], record P(next token = full_ids[t+1])
heat = np.zeros((len(hs_full), T_full - 1), dtype=np.float32)
with torch.no_grad():
    for layer_idx, h in enumerate(hs_full):
        probs = logit_lens(h, model)[0]              # (T_full, V)
        targets = full_ids[1:]                       # (T_full-1,)
        picked = probs[:-1].gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        heat[layer_idx] = picked.float().cpu().numpy()

# 4) Plot
token_strs = [tokenizer.decode([t.item()]).replace('\n', '\\n') for t in full_ids[1:]]
fig, ax = plt.subplots(figsize=(max(10, 0.55 * len(token_strs)), 0.32 * heat.shape[0] + 2))
im = ax.imshow(heat, aspect='auto', cmap='RdBu_r', vmin=0.0, vmax=1.0, origin='lower')
ax.set_xlabel('token position (predicting token t+1)')
ax.set_ylabel('layer  (0 = embedding, top = final)')
ax.set_xticks(range(len(token_strs)))
ax.set_xticklabels(token_strs, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(heat.shape[0]))
ax.set_yticklabels([label(i) for i in range(heat.shape[0])], fontsize=7)
ax.set_title(f'Logit Lens — P(target token) by layer\n{MODEL_ID}    prompt = {PROMPT!r}')
fig.colorbar(im, ax=ax, label='probability')
plt.tight_layout()
plt.savefig('logit_lens_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved logit_lens_heatmap.png')

## Caveat — and when to prefer Tuned Lens

The Logit Lens assumes the final layer-norm + unembed is a reasonable decoder for **every** layer.
That assumption breaks on post-2022 models (Llama-2+, Gemma-2, Qwen3.5+) which develop *rogue dimensions*
— a handful of residual-stream coordinates with massive scale that dominate `final_ln` activations but
carry no semantic signal. Symptoms: early layers look like garbage, middle layers fixate on a single
high-frequency token, top-of-stack suddenly snaps to the right answer.

**Fix**: [Tuned Lens](https://github.com/AlignmentResearch/tuned-lens) learns a per-layer affine
transformation `A_l h_l + b_l` so that lens-decoded logits from layer `l` match the true final-layer
logits. It's much smoother, reveals intermediate computation, and is what you want for any serious
interpretability claim. See notebook `20_tuned_lens.ipynb` in this series.

```bash
pip install tuned-lens
```

The raw Logit Lens here is still the right tool for a 5-minute sanity check, for intuition pumps in
a blog post, and as the baseline that Tuned Lens improves on.